# 05 · Emission integration, F1, the pooled scalar channel

Runs the F1 grid, joint-trained BKT with the pooled misconception scalar as a second, additive emission channel beside slip. The chains are fitted once under the chain of record, strong snap, and shared frozen across every row. Correctness is the only joint-training target, the channel input m at turn k is the chains' filtered state before that turn, solution row consumed by the chains, never by BKT, evaluation is paper-aligned, each dialogue's first scored turn updates the filter and is excluded from metrics, unseen KCs contribute 0.5.

**The rows:**

- Validity, beta pinned to 0, a like-for-like BKT refit whose distance from the frozen M1, 60.65 accuracy, 64.28 AUC, 55.60 f1, certifies the engine before any delta is read.
- The full grid, every connection, mastered, unmastered, both, crossed with every pooling method, max, noisy_or, top2_or, mean, each under a free beta and beta pinned to 1, capture at full strength.

**Reading order.** The validity row first, then the connection and pooling comparisons in sections 3 and 4. Fitted betas read as lower bounds on the capture rates, attenuated by chain measurement noise.

## 1. Setup

Chains fitted once, shared by every row.

In [1]:
import pandas as pd
from scripts.load_data import load_paper_filtered_data
from scripts.chain import TriggerChain
from scripts.misconception_chains import MisconceptionChains
from scripts.emission_integration_pooled import (
    CHAIN_OF_RECORD,
    EmissionIntegrationPooledMastered,
    EmissionIntegrationPooledUnmastered,
    EmissionIntegrationPooledBoth,
)

train_df = load_paper_filtered_data("data/mathdial_train.csv")
test_df = load_paper_filtered_data("data/mathdial_test.csv")

chains = MisconceptionChains(train_df, test_df, chain_class=TriggerChain,
                             chain_kwargs=dict(CHAIN_OF_RECORD))
chains.run()
chains.summary()

,chain,pi,onset,resolve,pP_i,pP_a,expected_dwell_turns,accuracy,tpr,tnr,auc,f1,informative_cells
0,comprehension,0.05,0.0,0.0323,0.02,0.97,31.0,0.6327,0.6705,0.3704,0.5639,0.7614,1285
1,relevance,0.05,0.0,0.0535,0.02,0.97,18.7,0.6370,0.6061,0.7222,0.6749,0.7101,135
2,principles,0.05,0.0,0.0113,0.02,0.97,88.8,0.8784,0.7742,0.9535,0.8593,0.8421,74
3,wrong_operation,0.05,0.0,0.0881,0.02,0.97,11.3,0.7036,0.5512,0.8435,0.7511,0.6403,1164
4,steps,0.05,0.0,0.0475,0.02,0.97,21.0,0.8516,0.5580,0.9599,0.7217,0.6696,512


## 2. Models

The full grid, every connection crossed with every pooling method, each under a free beta and beta pinned to 1, plus the validity row. Twenty-five fits, loop-built so each row's configuration is its name. The emission is additive throughout, competing risks with disjoint causes, incorrect probability is slip plus beta times m on the wired branches, clipped at the floor, so where capture exceeds a branch's success mass the branch saturates to certain incorrectness.

In [2]:
CONNECTIONS = {
    "mastered": EmissionIntegrationPooledMastered,
    "unmastered": EmissionIntegrationPooledUnmastered,
    "both": EmissionIntegrationPooledBoth,
}
POOLINGS = ["max", "noisy_or", "top2_or", "mean"]

ROWS = {"validity, beta=0": (EmissionIntegrationPooledMastered,
                             {"pooling": "max", "pin_beta": 0.0})}
for connection, cls in CONNECTIONS.items():
    for pooling in POOLINGS:
        ROWS[f"{connection}, {pooling}, free"] = (cls, {"pooling": pooling})
        ROWS[f"{connection}, {pooling}, beta=1"] = (cls, {"pooling": pooling,
                                                          "pin_beta": 1.0})

models = {}
for name, (cls, kwargs) in ROWS.items():
    model = cls(train_df, test_df, chains=chains, **kwargs)
    model.run()
    models[name] = model
    print(f"{name:28s} {model.metrics}")

validity, beta=0             {'accuracy': 0.6035, 'auc': 0.6397, 'f1': 0.5531, 'turns': 1985, 'beta_mastered': 0.0, 'beta_unmastered': None, 'pooling': 'max', 'connection': 'mastered'}
mastered, max, free          {'accuracy': 0.604, 'auc': 0.642, 'f1': 0.5493, 'turns': 1985, 'beta_mastered': 0.1546, 'beta_unmastered': None, 'pooling': 'max', 'connection': 'mastered'}
mastered, max, beta=1        {'accuracy': 0.6065, 'auc': 0.642, 'f1': 0.5803, 'turns': 1985, 'beta_mastered': 1.0, 'beta_unmastered': None, 'pooling': 'max', 'connection': 'mastered'}
mastered, noisy_or, free     {'accuracy': 0.606, 'auc': 0.6437, 'f1': 0.5516, 'turns': 1985, 'beta_mastered': 0.1565, 'beta_unmastered': None, 'pooling': 'noisy_or', 'connection': 'mastered'}
mastered, noisy_or, beta=1   {'accuracy': 0.6045, 'auc': 0.644, 'f1': 0.5809, 'turns': 1985, 'beta_mastered': 1.0, 'beta_unmastered': None, 'pooling': 'noisy_or', 'connection': 'mastered'}
mastered, top2_or, free      {'accuracy': 0.6045, 'auc': 0.6421,

## 3. Results table

The frozen M1 row is the anchor, its numbers from notebook 02. Deltas are against it.

In [3]:
import numpy as np

M1 = {"accuracy": 0.6065, "auc": 0.6428, "f1": 0.5560}

results = pd.DataFrame(
    [{"row": name, **models[name].metrics} for name in ROWS])
for metric in ("accuracy", "auc", "f1"):
    results[f"d_{metric}"] = (results[metric] - M1[metric]).round(4)
results = results.set_index("row")
results[["connection", "pooling", "beta_mastered", "beta_unmastered",
         "accuracy", "d_accuracy", "auc", "d_auc", "f1", "d_f1", "turns"]]

,connection,pooling,beta_mastered,beta_unmastered,accuracy,d_accuracy,auc,d_auc,f1,d_f1,turns
row,,,,,,,,,,,
"validity, beta=0",mastered,max,0.0000,NaN,0.6035,-0.0030,0.6397,-0.0031,0.5531,-0.0029,1985
"mastered, max, free",mastered,max,0.1546,NaN,0.6040,-0.0025,0.6420,-0.0008,0.5493,-0.0067,1985
"mastered, max, beta=1",mastered,max,1.0000,NaN,0.6065,0.0000,0.6420,-0.0008,0.5803,0.0243,1985
"mastered, noisy_or, free",mastered,noisy_or,0.1565,NaN,0.6060,-0.0005,0.6437,0.0009,0.5516,-0.0044,1985
"mastered, noisy_or, beta=1",mastered,noisy_or,1.0000,NaN,0.6045,-0.0020,0.6440,0.0012,0.5809,0.0249,1985
"mastered, top2_or, free",mastered,top2_or,0.1516,NaN,0.6045,-0.0020,0.6421,-0.0007,0.5496,-0.0064,1985
"mastered, top2_or, beta=1",mastered,top2_or,1.0000,NaN,0.6030,-0.0035,0.6431,0.0003,0.5777,0.0217,1985
"mastered, mean, free",mastered,mean,0.3777,NaN,0.5995,-0.0070,0.6401,-0.0027,0.5475,-0.0085,1985
"mastered, mean, beta=1",mastered,mean,1.0000,NaN,0.5985,-0.0080,0.6361,-0.0067,0.5303,-0.0257,1985


In [4]:
auc_pivot = results.reset_index()
auc_pivot["beta_mode"] = np.where(auc_pivot["row"].str.contains("beta=1"),
                                  "pinned", "free")
auc_pivot = auc_pivot[auc_pivot["row"] != "validity, beta=0"]
auc_pivot.pivot_table(index=["connection", "pooling"],
                      columns="beta_mode", values="auc").round(4)

beta_mode              free  pinned
connection pooling                 
both       max       0.6322  0.6012
           mean      0.6421  0.6347
           noisy_or  0.6344  0.6005
           top2_or   0.6335  0.6030
mastered   max       0.6420  0.6420
           mean      0.6401  0.6361
           noisy_or  0.6437  0.6440
           top2_or   0.6421  0.6431
unmastered max       0.6378  0.6339
           mean      0.6407  0.6380
           noisy_or  0.6381  0.6354
           top2_or   0.6388  0.6342

## 4. Deltas against the engine baseline

Every model against the validity row's own three metrics, the engine's beta-pinned-to-zero BKT, so each delta isolates what the channel configuration changed with code path, protocol, and optimizer held fixed. Sorted by AUC delta.

In [5]:
baseline = models["validity, beta=0"].metrics
deltas = pd.DataFrame([
    {"row": name,
     "d_accuracy": round(models[name].metrics["accuracy"]
                         - baseline["accuracy"], 4),
     "d_auc": round(models[name].metrics["auc"] - baseline["auc"], 4),
     "d_f1": round(models[name].metrics["f1"] - baseline["f1"], 4)}
    for name in ROWS if name != "validity, beta=0"])
deltas.set_index("row").sort_values("d_auc", ascending=False)

,d_accuracy,d_auc,d_f1
row,,,
"mastered, noisy_or, beta=1",0.0010,0.0043,0.0278
"mastered, noisy_or, free",0.0025,0.0040,-0.0015
"mastered, top2_or, beta=1",-0.0005,0.0034,0.0246
"mastered, top2_or, free",0.0010,0.0024,-0.0035
"both, mean, free",-0.0005,0.0024,-0.0034
"mastered, max, free",0.0005,0.0023,-0.0038
"mastered, max, beta=1",0.0030,0.0023,0.0272
"unmastered, mean, free",-0.0005,0.0010,-0.0065
"mastered, mean, free",-0.0040,0.0004,-0.0056


## 5. The co-elevation contrast

Test turns with the strongest chain live, stratified by whether the second-strongest is also live, observed incorrect rates compared. Under max pooling the two groups carry the same pooled risk, under noisy_or the two-live group carries more, so the observed rates indicate which combination rule the corpus follows. Underpowered if co-elevated turns are few, in which case say so rather than overclaim.

In [6]:
import numpy as np
from scripts.pooling import Pooling

headline = models["mastered, max, free"]
outcomes = headline.predict(headline.test)[
    ["dialogue_id", "turn_index", "correct"]].rename(
    columns={"turn_index": "position"})

records = []
for d, track in chains.predict_states(test_df).items():
    strongest = Pooling.max(track)
    second = Pooling.second_max(track)
    for pos in range(1, len(strongest)):
        records.append({"dialogue_id": d, "position": pos,
                        "strongest": strongest[pos], "second": second[pos]})
elev = pd.DataFrame(records).merge(outcomes, on=["dialogue_id", "position"])
elev = elev[elev["strongest"] >= 0.5]
elev["group"] = np.where(elev["second"] >= 0.5, "two live", "one live")

elev.groupby("group").agg(
    turns=("correct", "size"),
    incorrect_rate=("correct", lambda c: round(1 - c.mean(), 4)),
    mean_strongest=("strongest", "mean"),
).round(4)

,turns,incorrect_rate,mean_strongest
group,,,
one live,1892,0.5518,0.8309
two live,278,0.6007,0.8676


## 6. Interpretation ledger

- The validity row's distance from M1 bounds what initialization and the optimizer contribute, read every other delta net of it.
- The fitted betas are per-branch capture rates, fitted independently where both branches are wired, beta_mastered's gap from 0 is the channel's measured strength on the mastered branch, and attenuation from chain noise makes each a lower bound. On the both rows the split between the two betas locates where capture acts.
- If the unmastered rows match or trail the baseline while the mastered rows lead it, the channel's contribution is competence-side.
- Slip attribution, compare each KC's fitted slip here against the validity row's, the drop is how much of the baseline's slip the channel re-attributed, same optimizer both sides.

In [7]:
m2 = models["mastered, max, free"]
bkt = models["validity, beta=0"]
slips = pd.DataFrame([
    {"kc": kc, "m2_slip": round(p["slip"], 4),
     "bkt_slip": round(bkt.parameters[kc]["slip"], 4)}
    for kc, p in m2.parameters.items()])
slips["drop"] = (slips["bkt_slip"] - slips["m2_slip"]).round(4)
print(f"median slip drop {slips['drop'].median():+.4f} over {len(slips)} KCs, "
      f"beta_mastered {m2.metrics['beta_mastered']}")
slips.sort_values("drop", ascending=False).head(10)

median slip drop +0.0542 over 138 KCs, beta_mastered 0.1546


,kc,m2_slip,bkt_slip,drop
1,"Add and subtract within 1000, using concrete m...",0.0001,0.5531,0.5530
19,Compare two fractions with different numerator...,0.7219,0.9999,0.2780
65,Interpret multiplication as scaling (resizing)...,0.2383,0.5000,0.2617
61,Identify when two expressions are equivalent (...,0.1847,0.4429,0.2582
82,Recognize and represent proportional relations...,0.0903,0.3371,0.2468
52,Fluently add and subtract multi-digit whole nu...,0.1275,0.3690,0.2415
23,"Count to 120, starting at any number less than...",0.1975,0.4301,0.2326
51,Find whole-number quotients of whole numbers w...,0.6117,0.7850,0.1733
69,Make a line plot to display a data set of meas...,0.8503,0.9999,0.1496
62,Interpret a fraction as division of the numera...,0.8503,0.9999,0.1496


## 7. Why the channel did not improve prediction

The grid's verdict is a null on the primary metric, and two evaluation caveats frame every number here. Model selection happened on the test set, the chain configuration in notebook 03 and the 24 rows here were all compared on the same 1,985 turns, so the best row is optimistic and the deltas are descriptive. That cuts one way only, a best-of-24 selection that still lands within noise strengthens the null rather than weakening it. And the free rows in this saved run drew different per-KC initializations than the pinned and validity rows, an implementation confound since fixed, controlled same-initialization restarts across three seeds put the pooled free delta between +0.0008 and +0.0036, small and unstable, and a paired dialogue bootstrap on the best saved row spans zero, minus 0.0138 to plus 0.0214.

**Timing is the strongest explanation.** Review diagnostics on the same test turns make the gap concrete, the causal pre-turn pooled state predicts error at AUC 0.597, while two deliberately leaky diagnostics that peek at the current turn, the post-turn state and the count of current-turn P annotations, reach 0.761 and 0.862. The signal exists, overwhelmingly, at the same turn, and the legal pre-turn state cannot know that a belief will resolve or surface on the turn being predicted. Resolution turns show it sharpest, where the pre-turn state was still above 0.5 but the current annotation was A, correctness ran 80 to 93 per cent by family, the chain lowers its state after seeing the A, one turn too late to score that turn.

**The state is dialogue-level, and within dialogues it points the wrong way.** Between-dialogue differences carry 73 to 79 per cent of the pooled state's variance with lag-1 correlations near 0.75, the channel identifies difficult dialogues rather than difficult turns, and after dialogue-centering the within-dialogue correlation with error is slightly negative, around minus 0.11, higher pre-turn state inside a dialogue does not locally time the errors here.

**The channel is largely redundant with correctness history.** The pooled state correlates 0.30 with the baseline's own predicted error risk and only 0.06 with the baseline's residual error, the two predictors substantially overlap, prior wrong answers already lowered mastery on the dialogues the channel flags, so the emission is paid only for a residual that is nearly empty. This is overlap measured, not a causal mechanism established.

**What the pinned rows actually changed.** The f1 lift is real but its direction is the opposite of the earlier draft of this section, correctness is the positive class, and the best pinned row predicted more correct turns, 933 against the validity row's 821, closer to the 940 actually correct, joint refitting let the BKT parameters compensate for the subtracted channel term, so the movement is a threshold and calibration effect from the refit equilibrium, not a simple downward shift from capture, and not improved ranking.

**The remaining evidence is weak in the same direction.** The co-elevation contrast, 60.1 against 55.2 per cent incorrect, largely dissolves once the strongest-state level is adjusted for, an odds ratio near 1.18 at z 1.25. The fitted beta of 0.178 is an absolute probability decrement per unit of state, up to 18 points at m near 1, not a share of errors caused and not a guaranteed lower bound, and the slip drops, concentrated in a few KCs, are descriptive movements of a refit, part of them initialization noise in this saved run.

**Conclusion.** The misconception channel did not produce a reliable held-out AUC improvement. The annotations are strongly associated with correctness on the same turn, but the causally available pre-turn states are persistent, predominantly distinguish difficult dialogues, and are largely redundant with the correctness history BKT already conditions on. Pinned rows alter threshold behavior and f1 without establishing improved ranking, and because selection used the test set and the saved free rows carried an initialization confound, the fitted betas and slip movements are read as descriptive.

## 8. Notes

- Both sides of every comparison are paper-aligned by construction, per-KC predictions averaged to true turns, each dialogue's first scored turn updating the filter but excluded from metrics, exactly 0.5 classifying to 0, unseen KCs contributing 0.5, the notebook 02 protocol.
- The slip attribution in section 6 compares against the validity row's slips, same optimizer, only the channel differs, so the drop is purely the channel's re-attribution.
- Every model keeps its fitted parameters at `.parameters` and the shared frozen chains at `.chains`, nothing here refits the chains.
- The winner's row name, pooling, connection, and beta get logged with the pick, and the F2 and F3 faces reuse this notebook's shape.